# Data Visualization Seminar — Complete Colab Notebook

### Topics Covered
1. **Three-Dimensional Plotting in Matplotlib**
2. **Geographic Data with Basemap**
3. **Visualization with Seaborn**

**Datasets used (all built-in, no file upload needed):**

| Dataset | Source | Used for |
|---|---|---|
| `iris` | seaborn | 3D scatter, pair plots |
| `tips` | seaborn | KDE, joint plots, facet grids, factor plots |
| `flights` | seaborn | 3D surface, wireframe, heatmap |
| `planets` | seaborn | bar plots / count plots |
| `world_cities` | created in-notebook | Basemap geographic plotting |

> **How to use:** Open in Google Colab → `Runtime` → `Run all`. Section 2 (Basemap) needs a one-time install cell.

---

## 0. Setup — Imports and Dataset Loading

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 3D toolkit
from mpl_toolkits import mplot3d
from mpl_toolkits.mplot3d import Axes3D

# Display settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 100

print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("Matplotlib :", plt.matplotlib.__version__)
print("Seaborn    :", sns.__version__)

In [ ]:
# ---- Load all datasets ----
iris    = sns.load_dataset("iris")       # 150 flowers, 4 measurements, 3 species
tips    = sns.load_dataset("tips")       # 244 restaurant bills
flights = sns.load_dataset("flights")    # monthly airline passengers 1949-1960
planets = sns.load_dataset("planets")    # discovered exoplanets

print("iris   :", iris.shape)
print("tips   :", tips.shape)
print("flights:", flights.shape)
print("planets:", planets.shape)

iris.head()

In [ ]:
tips.head()

In [ ]:
# flights is in long format - pivot it into a 2D grid (needed for 3D surface plots)
flights_grid = flights.pivot(index="month", columns="year", values="passengers")
flights_grid

---
# 1. Three-Dimensional Plotting in Matplotlib

3D plots enable panna `mplot3d` toolkit import pannanum (Matplotlib-oda default-ah varum).
Basic idea: `projection='3d'` nu axes create pannanum.

```python
from mpl_toolkits import mplot3d
ax = plt.axes(projection='3d')
```

Available 3D plot types:
| Function | What it draws |
|---|---|
| `ax.plot3D()` | 3D line |
| `ax.scatter3D()` | 3D scatter points |
| `ax.contour3D()` | 3D contour / relief |
| `ax.plot_wireframe()` | Wireframe mesh |
| `ax.plot_surface()` | Filled surface |
| `ax.plot_trisurf()` | Triangulated surface |
| `ax.view_init()` | Change camera angle |


### 1.1 Creating a blank 3D axes

In [ ]:
fig = plt.figure(figsize=(6, 5))
ax = plt.axes(projection='3d')
ax.set_title("Empty 3D Axes")
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.show()

### 1.2 Three-Dimensional Points and Lines

Most basic 3D plot — `(x, y, z)` triples-la irundhu line or scatter.
`ax.plot3D()` line-kum, `ax.scatter3D()` points-kum.

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = plt.axes(projection='3d')

# --- Data for a three-dimensional LINE (a helix) ---
zline = np.linspace(0, 15, 1000)
xline = np.sin(zline)
yline = np.cos(zline)
ax.plot3D(xline, yline, zline, 'gray', linewidth=1.5)

# --- Data for three-dimensional SCATTERED points ---
zdata = 15 * np.random.random(100)
xdata = np.sin(zdata) + 0.1 * np.random.randn(100)
ydata = np.cos(zdata) + 0.1 * np.random.randn(100)
ax.scatter3D(xdata, ydata, zdata, c=zdata, cmap='Greens')

ax.set_title("3D Line + Scatter (Helix)")
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
plt.show()

# NOTE: scatter points automatically get transparency adjusted
#       to give a sense of depth on the page.

#### 1.2(b) Same idea, but with a REAL dataset — Iris 3D scatter

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection='3d')

colors = {'setosa': 'tab:blue', 'versicolor': 'tab:orange', 'virginica': 'tab:green'}

for species, group in iris.groupby('species'):
    ax.scatter3D(group['sepal_length'],
                 group['sepal_width'],
                 group['petal_length'],
                 s=45, alpha=0.8,
                 color=colors[species],
                 label=species)

ax.set_xlabel('Sepal Length (cm)')
ax.set_ylabel('Sepal Width (cm)')
ax.set_zlabel('Petal Length (cm)')
ax.set_title("Iris Dataset — 3D Scatter by Species")
ax.legend()
ax.view_init(20, 45)   # elevation=20, azimuth=45
plt.show()

# Observation: 'setosa' completely separates from the other two along petal_length.

### 1.3 Three-Dimensional Contour Plots

`ax.contour3D()` — 2D contour madhiri, aana Z data-va height-ah plot pannum.
Input data 2D regular grid-ah irukanum (`np.meshgrid` use pannanum).

In [ ]:
# Define a 3D sinusoidal function
def f(x, y):
    return np.sin(np.sqrt(x ** 2 + y ** 2))

x = np.linspace(-6, 6, 30)
y = np.linspace(-6, 6, 30)
X, Y = np.meshgrid(x, y)     # create the 2D regular grid
Z = f(X, Y)                  # evaluate Z at each grid point

print("Grid shapes -> X:", X.shape, " Y:", Y.shape, " Z:", Z.shape)

fig = plt.figure(figsize=(7, 6))
ax = plt.axes(projection='3d')
ax.contour3D(X, Y, Z, 50, cmap='binary')   # 50 = number of contour levels
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title("3D Contour Plot — sin(sqrt(x²+y²))")
plt.show()

#### 1.3(b) Changing the viewing angle with `view_init()`

In [ ]:
# view_init(elevation, azimuth) -> rotate the camera
fig = plt.figure(figsize=(12, 5))

for i, (elev, azim) in enumerate([(60, 35), (10, 120)]):
    ax = fig.add_subplot(1, 2, i+1, projection='3d')
    ax.contour3D(X, Y, Z, 50, cmap='binary')
    ax.view_init(elev, azim)
    ax.set_title(f"view_init(elev={elev}, azim={azim})")

plt.tight_layout()
plt.show()

### 1.4 Wireframes and Surface Plots

Rendum gridded data-la work aagum.
- **Wireframe** — mesh lines mattum
- **Surface** — ovvoru face-um filled polygon, colormap add pannina topology clear-ah theriyum

In [ ]:
fig = plt.figure(figsize=(13, 5))

# ---- WIREFRAME ----
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_wireframe(X, Y, Z, color='black', linewidth=0.6)
ax1.set_title('Wireframe')

# ---- SURFACE ----
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
surf = ax2.plot_surface(X, Y, Z, rstride=1, cstride=1,
                        cmap='viridis', edgecolor='none')
ax2.set_title('Surface')
fig.colorbar(surf, ax=ax2, shrink=0.5)

plt.tight_layout()
plt.show()

#### 1.4(b) Surface plot on a REAL dataset — Flights (Passengers over Year × Month)

`flights_grid` already 2D grid-ah irukku (12 months × 12 years), so direct-ah surface plot podalam.

In [ ]:
Zf = flights_grid.values                                  # 12 months x 12 years
Xf, Yf = np.meshgrid(np.arange(Zf.shape[1]),              # year index
                     np.arange(Zf.shape[0]))              # month index

fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection='3d')
surf = ax.plot_surface(Xf, Yf, Zf, cmap='viridis', edgecolor='none')

ax.set_xticks(np.arange(0, Zf.shape[1], 2))
ax.set_xticklabels(flights_grid.columns[::2])
ax.set_yticks(np.arange(Zf.shape[0]))
ax.set_yticklabels(flights_grid.index, fontsize=8)

ax.set_xlabel('Year'); ax.set_ylabel('Month'); ax.set_zlabel('Passengers')
ax.set_title("Airline Passengers 1949–1960 (3D Surface)")
ax.view_init(30, -120)
fig.colorbar(surf, shrink=0.5, label='Passengers')
plt.show()

# Observation: overall upward trend across years + a repeating summer peak every year.

### 1.5 Surface Triangulations

Sila neram evenly-sampled grid create pannradhu kastam (random / irregular points).
Andha maadhiri situations-la **triangulation-based** plots useful — `plot_trisurf()`.

In [ ]:
# Generate RANDOM (non-grid) sample points in a disc
theta = 2 * np.pi * np.random.random(1000)
r = 6 * np.random.random(1000)
x_t = np.ravel(r * np.sin(theta))
y_t = np.ravel(r * np.cos(theta))
z_t = f(x_t, y_t)

fig = plt.figure(figsize=(13, 5))

# ---- Step 1: raw scattered points ----
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.scatter(x_t, y_t, z_t, c=z_t, cmap='viridis', linewidth=0.5, s=8)
ax1.set_title('Randomly Sampled Points')

# ---- Step 2: triangulated surface from those same points ----
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_trisurf(x_t, y_t, z_t, cmap='viridis', edgecolor='none')
ax2.set_title('Surface Triangulation (plot_trisurf)')

plt.tight_layout()
plt.show()

### 1.6 Bonus — 3D Bar Plot using the Tips dataset

`ax.bar3d()` use panni categorical data-vum 3D-la kaatalam.

In [ ]:
# Average total_bill for each (day, sex) combination
pivot = tips.pivot_table(index='day', columns='sex',
                         values='total_bill', aggfunc='mean', observed=True)
print(pivot)

fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection='3d')

xpos, ypos = np.meshgrid(np.arange(pivot.shape[1]), np.arange(pivot.shape[0]))
xpos = xpos.ravel(); ypos = ypos.ravel()
zpos = np.zeros_like(xpos, dtype=float)
dz = pivot.values.ravel()

ax.bar3d(xpos, ypos, zpos, 0.5, 0.5, dz, shade=True, color='steelblue')
ax.set_xticks(np.arange(pivot.shape[1]) + 0.25)
ax.set_xticklabels(pivot.columns)
ax.set_yticks(np.arange(pivot.shape[0]) + 0.25)
ax.set_yticklabels(pivot.index)
ax.set_zlabel('Mean Total Bill ($)')
ax.set_title('Average Bill by Day and Sex (3D Bars)')
plt.show()

---
# 2. Geographic Data with Basemap

Basemap = Matplotlib-oda toolkit for plotting data on maps. `mpl_toolkits` namespace-la irukum.

> ⚠️ **Important:** Basemap is deprecated and does **not** ship with Colab.
> Kizhe irukura install cell-ah **mudhalla run pannunga** (~2–3 minutes).
> Install fail aana, **Section 2B (Cartopy)** use pannunga — same maps, modern library.

### 2.0 Install Basemap (run once)

In [ ]:
# Run this cell ONCE. It takes 2-3 minutes.
!apt-get -qq install -y libgeos-dev > /dev/null
!pip install -q basemap basemap-data basemap-data-hires

print("Install finished. Restart runtime if import fails.")

In [ ]:
# Verify the install
try:
    from mpl_toolkits.basemap import Basemap
    BASEMAP_OK = True
    print("Basemap imported successfully")
except Exception as e:
    BASEMAP_OK = False
    print("Basemap NOT available ->", e)
    print("Use Section 2B (Cartopy) instead.")

### 2.1 A Globe — Orthographic Projection with Blue Marble

`bluemarble()` — NASA-oda satellite image-ah globe mேl project pannum.

In [ ]:
from mpl_toolkits.basemap import Basemap

plt.figure(figsize=(8, 8))
m = Basemap(projection='ortho', resolution=None, lat_0=50, lon_0=-100)
m.bluemarble(scale=0.5)
plt.title("Orthographic Projection — NASA Blue Marble", pad=20)
plt.show()

### 2.2 Helper function `draw_map()` — shaded relief + lat/lon grid lines

Idha oru mura define pannitta, ella projection-kum reuse pannalam.

In [ ]:
from itertools import chain

def draw_map(m, scale=0.2):
    # draw a shaded-relief image as the background
    m.shadedrelief(scale=scale)

    # lats and lons are returned as dictionaries
    lats = m.drawparallels(np.linspace(-90, 90, 13))
    lons = m.drawmeridians(np.linspace(-180, 180, 13))

    # keys contain the plt.Line2D instances
    lat_lines = chain(*(tup[1][0] for tup in lats.items()))
    lon_lines = chain(*(tup[1][0] for tup in lons.items()))
    all_lines = chain(lat_lines, lon_lines)

    # cycle through these lines and set the desired style
    for line in all_lines:
        line.set(linestyle='-', alpha=0.3, color='r')

print("draw_map() ready")

### 2.3 ETOPO relief map of a particular area (North America)

`etopo()` — topographical features land-lum ocean-lum kaatum.

In [ ]:
fig = plt.figure(figsize=(8, 8))
m = Basemap(projection='lcc', resolution=None,
            width=8E6, height=8E6,
            lat_0=45, lon_0=-100)
m.etopo(scale=0.5, alpha=0.5)
plt.title("ETOPO Relief — North America (Lambert Conformal Conic)")
plt.show()

### 2.4 Map Projections

Basemap-la several dozen projections irukku, ovvondrum oru short code.
Naalu main families:

| Family | Example code | Character |
|---|---|---|
| Cylindrical | `'cyl'`, `'merc'`, `'cea'` | Lat/lon → straight horizontal/vertical lines; poles-la heavy distortion |
| Pseudo-cylindrical | `'moll'`, `'sinu'`, `'robin'` | Meridians curved; area preserved |
| Perspective | `'ortho'`, `'gnom'`, `'stere'` | Space-la irundhu photo eduthaa madhiri; half globe mattum |
| Conic | `'lcc'`, `'eqdc'`, `'aea'` | Cone-la project panni unroll pannradhu; local accuracy nalladhu |


#### 2.4(a) Cylindrical Projection

In [ ]:
fig = plt.figure(figsize=(9, 6), edgecolor='w')
m = Basemap(projection='cyl', resolution=None,
            llcrnrlat=-90,  urcrnrlat=90,
            llcrnrlon=-180, urcrnrlon=180)
draw_map(m)
plt.title("Cylindrical Projection (projection='cyl')")
plt.show()

#### 2.4(b) Pseudo-cylindrical — Mollweide

In [ ]:
fig = plt.figure(figsize=(9, 6), edgecolor='w')
m = Basemap(projection='moll', resolution=None, lat_0=0, lon_0=0)
draw_map(m)
plt.title("Mollweide Projection (projection='moll') — area preserving")
plt.show()

#### 2.4(c) Perspective — Orthographic

In [ ]:
fig = plt.figure(figsize=(8, 8))
m = Basemap(projection='ortho', resolution=None, lat_0=50, lon_0=0)
draw_map(m)
plt.title("Orthographic Projection (projection='ortho')")
plt.show()

#### 2.4(d) Conic — Lambert Conformal Conic

In [ ]:
fig = plt.figure(figsize=(8, 8))
m = Basemap(projection='lcc', resolution=None,
            lon_0=0, lat_0=50, lat_1=45, lat_2=55,
            width=1.6E7, height=1.2E7)
draw_map(m)
plt.title("Lambert Conformal Conic (projection='lcc')")
plt.show()

### 2.5 Drawing a Map Background

Basemap-la borders, coastlines, political boundaries ellam draw panna functions irukku.

**Physical boundaries and bodies of water**
- `drawcoastlines()` — continental coast lines
- `drawlsmask()` — land/sea mask
- `drawmapboundary()` — map boundary + ocean fill colour
- `drawrivers()` — rivers
- `fillcontinents()` — fill continents (and optionally lakes)

**Political boundaries**
- `drawcountries()`, `drawstates()`, `drawcounties()`

**Map features**
- `drawgreatcircle()`, `drawparallels()`, `drawmeridians()`, `drawmapscale()`

**Whole-globe images**
- `bluemarble()`, `shadedrelief()`, `etopo()`, `warpimage()`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, res in zip(axes, ['l', 'c']):
    m = Basemap(projection='lcc', resolution=res,
                lat_0=45, lon_0=-100, width=8E6, height=8E6, ax=ax)
    m.drawmapboundary(fill_color='#a6cee3')      # ocean colour
    m.fillcontinents(color='#e6d8ad', lake_color='#a6cee3')
    m.drawcoastlines(color='gray', linewidth=0.5)
    m.drawcountries(color='gray', linewidth=0.7)
    m.drawstates(color='gray', linewidth=0.3)
    ax.set_title(f"resolution='{res}'")

plt.tight_layout()
plt.show()

### 2.6 Plotting Data on Maps — with a real dataset

Basemap-oda real power: map background mேல data over-plot pannradhu.

Map-specific methods: `contour()/contourf()`, `imshow()`, `pcolor()/pcolormesh()`,
`plot()`, `scatter()`, `quiver()`, `barbs()`, `drawgreatcircle()`.

Inga naama oru **world cities dataset** create panni, population-ah bubble size-ah plot panrom.

In [ ]:
# ---- Dataset: major world cities with coordinates and population (millions) ----
world_cities = pd.DataFrame({
    'city': ['Tokyo','Delhi','Shanghai','Sao Paulo','Mexico City','Cairo','Mumbai',
             'Beijing','Dhaka','Osaka','New York','Karachi','Chennai','Buenos Aires',
             'Istanbul','Lagos','Moscow','Paris','London','Los Angeles','Jakarta',
             'Bangkok','Seoul','Lima','Bogota','Sydney','Toronto','Chicago'],
    'lat':  [35.68, 28.61, 31.23,-23.55, 19.43, 30.04, 19.08,
             39.90, 23.81, 34.69, 40.71, 24.86, 13.08,-34.60,
             41.01,  6.52, 55.76, 48.86, 51.51, 34.05, -6.21,
             13.76, 37.57,-12.05,  4.71,-33.87, 43.65, 41.88],
    'lon':  [139.69, 77.21,121.47,-46.63,-99.13, 31.24, 72.88,
             116.41, 90.41,135.50,-74.01, 67.01, 80.27,-58.38,
              28.98,  3.38, 37.62,  2.35, -0.13,-118.24,106.85,
             100.50,126.98,-77.04,-74.07,151.21,-79.38,-87.63],
    'pop':  [37.4, 32.9, 29.2, 22.6, 22.3, 22.2, 21.3,
             21.8, 23.2, 19.0, 18.9, 17.2, 11.8, 15.4,
             15.8, 15.9, 12.7, 11.2,  9.6, 12.5, 11.2,
             10.9,  9.9, 11.2, 11.3,  5.3,  6.4,  8.9]
})
print(world_cities.shape)
world_cities.head()

In [ ]:
fig = plt.figure(figsize=(14, 8))

m = Basemap(projection='robin', resolution='l', lat_0=0, lon_0=0)
m.drawmapboundary(fill_color='#a6cee3')
m.fillcontinents(color='#e6d8ad', lake_color='#a6cee3')
m.drawcoastlines(color='gray', linewidth=0.4)
m.drawcountries(color='gray', linewidth=0.4)

# Convert lat/lon -> map projection coordinates
x, y = m(world_cities['lon'].values, world_cities['lat'].values)

sc = m.scatter(x, y,
               s=world_cities['pop'] * 12,     # bubble size = population
               c=world_cities['pop'],          # colour  = population
               cmap='Reds', alpha=0.75,
               edgecolor='k', linewidth=0.5, zorder=5)

# Label the five largest cities
for _, row in world_cities.nlargest(5, 'pop').iterrows():
    xi, yi = m(row['lon'], row['lat'])
    plt.text(xi, yi + 4e5, row['city'], fontsize=9, ha='center',
             fontweight='bold', zorder=6)

plt.colorbar(sc, label='Population (millions)', shrink=0.6)
plt.title("World's Largest Cities by Population (Robinson Projection)", fontsize=14)
plt.show()

### 2.7 Great Circle — shortest path between two cities

`drawgreatcircle()` — globe-la rendu points-ku idaiyila irukura shortest route.

In [ ]:
fig = plt.figure(figsize=(14, 8))
m = Basemap(projection='robin', resolution='l', lat_0=0, lon_0=0)
m.drawmapboundary(fill_color='#a6cee3')
m.fillcontinents(color='#e6d8ad', lake_color='#a6cee3')
m.drawcoastlines(color='gray', linewidth=0.4)

# Flight routes starting from Chennai
routes = [('Chennai', 'London'), ('Chennai', 'Tokyo'),
          ('Chennai', 'New York'), ('Chennai', 'Sydney')]

coords = world_cities.set_index('city')[['lat', 'lon']].to_dict('index')

for src, dst in routes:
    m.drawgreatcircle(coords[src]['lon'], coords[src]['lat'],
                      coords[dst]['lon'], coords[dst]['lat'],
                      linewidth=2, color='crimson', alpha=0.8)

for city in set([c for r in routes for c in r]):
    xi, yi = m(coords[city]['lon'], coords[city]['lat'])
    m.scatter(xi, yi, s=60, color='darkblue', zorder=5)
    plt.text(xi, yi + 4e5, city, fontsize=9, ha='center', fontweight='bold')

plt.title("Great Circle Routes from Chennai", fontsize=14)
plt.show()

---
## 2B. Cartopy Alternative (if Basemap install fails)

Basemap deprecated-ah irukkradhaala, **Cartopy** modern replacement. Same maps, easy install.
Section 2 work aagala na, intha section-ah use pannunga.

In [ ]:
!pip install -q cartopy
print("cartopy installed")

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Four projections, one figure — same families as Basemap
projections = [
    ("Cylindrical (PlateCarree)", ccrs.PlateCarree()),
    ("Pseudo-cylindrical (Mollweide)", ccrs.Mollweide()),
    ("Perspective (Orthographic)", ccrs.Orthographic(0, 50)),
    ("Conic (LambertConformal)", ccrs.LambertConformal()),
]

fig = plt.figure(figsize=(14, 10))
for i, (name, proj) in enumerate(projections, 1):
    ax = fig.add_subplot(2, 2, i, projection=proj)
    ax.add_feature(cfeature.OCEAN, facecolor='#a6cee3')
    ax.add_feature(cfeature.LAND,  facecolor='#e6d8ad')
    ax.coastlines(linewidth=0.5)
    ax.gridlines(linestyle='--', alpha=0.4)
    ax.set_global()
    ax.set_title(name)

plt.tight_layout()
plt.show()

In [ ]:
# Plotting the SAME world_cities dataset with Cartopy
fig = plt.figure(figsize=(14, 8))
ax = plt.axes(projection=ccrs.Robinson())
ax.add_feature(cfeature.OCEAN, facecolor='#a6cee3')
ax.add_feature(cfeature.LAND,  facecolor='#e6d8ad')
ax.coastlines(linewidth=0.4)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.set_global()

sc = ax.scatter(world_cities['lon'], world_cities['lat'],
                s=world_cities['pop'] * 12,
                c=world_cities['pop'], cmap='Reds',
                alpha=0.75, edgecolor='k', linewidth=0.5,
                transform=ccrs.PlateCarree(), zorder=5)

plt.colorbar(sc, ax=ax, label='Population (millions)', shrink=0.6)
plt.title("World's Largest Cities — Cartopy Version", fontsize=14)
plt.show()

---
# 3. Visualization with Seaborn

Seaborn-oda main idea: **high-level commands** kuduthu statistical plots easy-ah create pannalam,
and konjam statistical model fitting kuda pannalam.

Matplotlib-oda default style-ah azhaga maathradhukku `sns.set_theme()` podanum.

In [ ]:
sns.set_theme(style="darkgrid")
print("Seaborn theme applied")

### 3.1 Histograms, KDE, and Densities

**KDE (Kernel Density Estimation)** — histogram-oda smooth version.
Modern function: `sns.kdeplot()`.

> Note: `sns.distplot()` deprecated aagiduchu. Ippo `sns.histplot()` (histogram + optional KDE)
> allathu `sns.displot()` use pannanum.

In [ ]:
# Generate a correlated 2D dataset (as in the reference material)
data = np.random.multivariate_normal([0, 0], [[5, 2], [2, 2]], size=2000)
data = pd.DataFrame(data, columns=['x', 'y'])

plt.figure(figsize=(8, 5))
for col in 'xy':
    sns.kdeplot(data[col], fill=True, label=col)
plt.title("KDE Plot — smooth density estimate")
plt.legend()
plt.show()

In [ ]:
# Histogram + KDE combined (the modern replacement for distplot)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(data['x'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title("histplot(x, kde=True)")

sns.histplot(data['y'], kde=True, ax=axes[1], color='coral')
axes[1].set_title("histplot(y, kde=True)")

plt.tight_layout()
plt.show()

In [ ]:
# Same thing on a REAL dataset — distribution of restaurant bills
plt.figure(figsize=(8, 5))
sns.histplot(data=tips, x='total_bill', hue='sex', kde=True,
             element='step', palette='Set1')
plt.title("Distribution of Total Bill by Sex (Tips dataset)")
plt.show()

In [ ]:
# TWO-DIMENSIONAL KDE — pass both variables
plt.figure(figsize=(7, 6))
sns.kdeplot(data=data, x='x', y='y', fill=True, cmap='Blues', levels=15)
plt.title("2D KDE — joint density of x and y")
plt.show()

### 3.2 Joint Distributions

`sns.jointplot()` — joint distribution + rendu marginal distributions-um sernthu kaatum.

In [ ]:
# Hex-bin version
g = sns.jointplot(data=tips, x='total_bill', y='tip', kind='hex', color='steelblue')
g.figure.suptitle("Joint Distribution — hexbin", y=1.02)
plt.show()

In [ ]:
# KDE and regression versions
for kind in ['kde', 'reg']:
    g = sns.jointplot(data=tips, x='total_bill', y='tip', kind=kind, height=5)
    g.figure.suptitle(f"Joint Distribution — kind='{kind}'", y=1.02)
    plt.show()

### 3.3 Pair Plots

Joint plots-ah higher dimensions-ku generalize pannina → **pair plot**.
Ella pairs of columns-um oruthuku edhirah plot aagum — correlations explore panna semma useful.

In [ ]:
g = sns.pairplot(iris, hue='species', height=2.2, diag_kind='kde')
g.figure.suptitle("Pair Plot — Iris Dataset", y=1.02, fontsize=14)
plt.show()

# Observation: petal_length and petal_width separate the three species most cleanly.

In [ ]:
# Pair plot on the tips dataset
g = sns.pairplot(tips, hue='sex', height=2.2,
                 vars=['total_bill', 'tip', 'size'])
g.figure.suptitle("Pair Plot — Tips Dataset", y=1.02, fontsize=14)
plt.show()

### 3.4 Faceted Histograms

Sometimes best way to view data = histograms of **subsets**.
`sns.FacetGrid` idha romba simple-ah aakkidum.

Inga tips dataset use panrom — restaurant staff-ku kidaikira tip percentage-ah
sex and time (Lunch/Dinner) based-ah split panni paarkarom.

In [ ]:
# Create the tip percentage column
tips_f = tips.copy()
tips_f['tip_pct'] = 100 * tips_f['tip'] / tips_f['total_bill']

g = sns.FacetGrid(tips_f, row="sex", col="time", margin_titles=True, height=3)
g.map(plt.hist, "tip_pct", bins=np.linspace(0, 40, 15), color='steelblue')
g.set_axis_labels("Tip %", "Count")
g.figure.suptitle("Faceted Histograms — Tip % by Sex and Time", y=1.04, fontsize=13)
plt.show()

In [ ]:
# The same with the modern displot() interface
sns.displot(data=tips_f, x='tip_pct', col='time', row='smoker',
            bins=np.linspace(0, 40, 15), height=3, facet_kws={'margin_titles': True})
plt.show()

### 3.5 Factor Plots (now `catplot`)

Factor plot — oru parameter-oda distribution-ah, vera oru parameter define pannina bins-la paakalam.

> `sns.factorplot()` renamed to `sns.catplot()`. `kind=` argument-la
> `'box'`, `'violin'`, `'bar'`, `'strip'`, `'swarm'`, `'count'` ellam kudukalam.

In [ ]:
g = sns.catplot(data=tips, x='day', y='total_bill', hue='sex',
                kind='box', height=5, aspect=1.4, palette='Set2')
g.set_axis_labels("Day", "Total Bill ($)")
g.figure.suptitle("Factor Plot (box) — Total Bill by Day and Sex", y=1.03)
plt.show()

In [ ]:
# Compare several kinds side by side
for kind in ['violin', 'bar', 'swarm']:
    g = sns.catplot(data=tips, x='day', y='total_bill', hue='sex',
                    kind=kind, height=4, aspect=1.5, palette='Set2')
    g.figure.suptitle(f"catplot(kind='{kind}')", y=1.04)
    plt.show()

### 3.6 Bar Plots / Time Series

Time series-ah `catplot(kind='count')` use panni plot pannalam.
Inga **planets** dataset — ovvoru varusham evlo exoplanets discover aachu nu count.

In [ ]:
g = sns.catplot(data=planets, x='year', kind='count',
                height=5, aspect=2.5, color='steelblue')
g.set_xticklabels(step=2)
g.set_axis_labels("Year", "Number of Planets Discovered")
g.figure.suptitle("Exoplanet Discoveries per Year", y=1.03, fontsize=13)
plt.show()

In [ ]:
# Break it down by discovery method (only methods with many discoveries)
top_methods = planets['method'].value_counts().head(3).index
subset = planets[planets['method'].isin(top_methods)]

g = sns.catplot(data=subset, x='year', kind='count', hue='method',
                height=5, aspect=2.5, palette='Set1')
g.set_xticklabels(step=2)
g.figure.suptitle("Exoplanet Discoveries per Year by Method", y=1.03)
plt.show()

### 3.7 Bonus — Heatmap and Regression Plot

In [ ]:
# Heatmap of the flights grid (same data used for the 3D surface earlier)
plt.figure(figsize=(10, 6))
sns.heatmap(flights_grid, cmap='YlGnBu', annot=False, cbar_kws={'label': 'Passengers'})
plt.title("Airline Passengers Heatmap (Month × Year)")
plt.show()

In [ ]:
# Linear regression fit with confidence band
g = sns.lmplot(data=tips, x='total_bill', y='tip', hue='sex',
               height=5, aspect=1.4, palette='Set1')
g.set_axis_labels("Total Bill ($)", "Tip ($)")
g.figure.suptitle("Regression — Tip vs Total Bill", y=1.03)
plt.show()

# Correlation check
print("Correlation (total_bill vs tip):",
      round(tips['total_bill'].corr(tips['tip']), 3))

---
# Summary

| Section | Key functions | Dataset used |
|---|---|---|
| **3D Plotting** | `plot3D`, `scatter3D`, `contour3D`, `plot_wireframe`, `plot_surface`, `plot_trisurf`, `bar3d`, `view_init` | iris, flights, tips |
| **Basemap** | `Basemap(projection=...)`, `bluemarble`, `etopo`, `shadedrelief`, `drawcoastlines`, `fillcontinents`, `scatter`, `drawgreatcircle` | world_cities |
| **Seaborn** | `kdeplot`, `histplot`, `jointplot`, `pairplot`, `FacetGrid`, `catplot`, `heatmap`, `lmplot` | iris, tips, planets, flights |

### Key takeaways
- 3D plots-ku data **grid format**-la (`np.meshgrid`) irukanum — irregular points-ku `plot_trisurf` use pannunga.
- `view_init(elev, azim)` illama 3D plot-la neraya details miss aagum — always angle adjust pannunga.
- Basemap deprecated — new projects-ku **Cartopy** recommended.
- Seaborn-oda old names (`distplot`, `factorplot`) deprecated — `histplot`/`displot` and `catplot` use pannunga.
